# Track A: Leave-One-Facility-Out CV (Colab GPU version)

Ported from `lofo_track_a.py` and `lofo_track_a_aug.py`. This is the paper's headline Track A generalization result: a fresh `Detector3` model is trained from scratch once per held-out plant facility (20 folds), then a second pass repeats all 20 folds with light spatial augmentation. That's 20-40 full training runs -- by far the most GPU-hungry Track A workload in the repo, so this is where a Colab T4 helps most.

Logic (model, folds, seeds, epochs, augmentation) is unchanged from the originals; only I/O is adapted for Colab.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

**Data:** same `threech_data.zip` used by `train_3channel_colab.ipynb` -- upload it below or point at a Drive copy.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Get the data onto the Colab VM

Run ONE of the next two cells.

In [ ]:
# Option A: direct upload
from google.colab import files
uploaded = files.upload()  # select threech_data.zip
DATA_ZIP = next(iter(uploaded.keys()))

In [ ]:
# Option B: Google Drive (skip if you used Option A above)
from google.colab import drive
drive.mount('/content/drive')
DATA_ZIP = '/content/drive/MyDrive/co2-emission-estimation/threech_data.zip'

In [ ]:
import zipfile, os
os.makedirs('/content/work', exist_ok=True)
with zipfile.ZipFile(DATA_ZIP) as zf:
    zf.extractall('/content/work')

DATA_ROOT = '/content/work/data/threech'
assert os.path.isdir(DATA_ROOT), f"expected {DATA_ROOT} to exist after extraction"
for sub in ('positive', 'hard_negative', 'negative'):
    n = len([f for f in os.listdir(f'{DATA_ROOT}/{sub}') if f.endswith('.npy')])
    print(f'{sub}: {n} tiles')

## 2. Shared model + LOFO fold machinery (unchanged from `lofo_track_a.py`)

In [ ]:
import numpy as np, glob, os, re, random, json, time, torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 30
_MONTH_SUFFIX = re.compile(r"_\d{4}_\d{2}\.npy$")


def load_folder(path, label):
    X, y, groups = [], [], []
    for f in sorted(glob.glob(f"{path}/*.npy")):
        arr = np.load(f).astype(np.float32)
        X.append(arr); y.append(label)
        groups.append(_MONTH_SUFFIX.sub("", os.path.basename(f)))
    return X, y, groups


class Detector3(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(64, 2))

    def forward(self, x):
        return self.net(x)


class AugmentedTiles(Dataset):
    """Applies a random dihedral-group-4 transform (identity / hflip / vflip
    / 90-deg rotation) per __getitem__ call, re-sampled every epoch since
    DataLoader re-iterates the Dataset each epoch."""
    def __init__(self, X, y):
        self.X = X  # (N, 3, 64, 64) normalized
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        tile = self.X[idx]
        choice = random.randint(0, 3)
        if choice == 1:
            tile = np.flip(tile, axis=1)
        elif choice == 2:
            tile = np.flip(tile, axis=2)
        elif choice == 3:
            tile = np.rot90(tile, k=1, axes=(1, 2))
        return torch.tensor(np.ascontiguousarray(tile)), self.y[idx]


def train_and_eval(Xtr, ytr, Xte, seed, augment=False):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    Xtr_arr = np.stack(Xtr).astype(np.float32)
    mean = np.array([Xtr_arr[:, c].mean() for c in range(3)], dtype=np.float32)
    std = np.array([Xtr_arr[:, c].std() + 1e-12 for c in range(3)], dtype=np.float32)
    Xtr_norm = (Xtr_arr - mean[None, :, None, None]) / std[None, :, None, None]

    ytr_arr = np.array(ytr, dtype=np.int64)
    if augment:
        tr_dl = DataLoader(AugmentedTiles(Xtr_norm, torch.tensor(ytr_arr)),
                            batch_size=16, shuffle=True)
    else:
        tr_dl = DataLoader(TensorDataset(torch.tensor(Xtr_norm), torch.tensor(ytr_arr)),
                            batch_size=16, shuffle=True)

    model = Detector3().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    lossf = nn.CrossEntropyLoss()
    for _ in range(EPOCHS):
        model.train()
        for xb, yb in tr_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); lossf(model(xb), yb).backward(); opt.step()

    Xte_arr = np.stack(Xte).astype(np.float32)
    Xte_norm = (Xte_arr - mean[None, :, None, None]) / std[None, :, None, None]

    model.eval()
    with torch.no_grad():
        xb = torch.tensor(Xte_norm, dtype=torch.float32).to(DEVICE)
        pred = model(xb).argmax(1).cpu().numpy()
    recall = float((pred == 1).mean())
    return recall


def facility_fold_indices(groups, held_out):
    """
    Split per-tile facility-group labels into (train_idx, test_idx) for one
    LOFO fold: every tile whose group != held_out goes to train, every tile
    whose group == held_out goes to test.
    """
    tr_idx = [j for j, g in enumerate(groups) if g != held_out]
    te_idx = [j for j, g in enumerate(groups) if g == held_out]
    return tr_idx, te_idx


Xp, yp, gp = load_folder(f"{DATA_ROOT}/positive", 1)
Xh, yh, gh = load_folder(f"{DATA_ROOT}/hard_negative", 0)
Xr, yr, gr = load_folder(f"{DATA_ROOT}/negative", 0)
plant_groups = sorted(set(gp))
print(f"{len(plant_groups)} plant facilities, {len(Xh)} hard negatives + {len(Xr)} rural negatives held fixed in train each fold")

## 3a. Baseline LOFO (20 folds, no augmentation) -- from `lofo_track_a.py`

In [ ]:
OUT_DIR = '/content/work'

results = []
t0 = time.time()
for i, held_out in enumerate(plant_groups):
    tr_idx, te_idx = facility_fold_indices(gp, held_out)
    Xtr = [Xp[j] for j in tr_idx] + Xh + Xr
    ytr = [yp[j] for j in tr_idx] + yh + yr
    Xte = [Xp[j] for j in te_idx]

    recall = train_and_eval(Xtr, ytr, Xte, seed=i, augment=False)
    results.append({"plant": held_out, "n_tiles": len(te_idx), "recall": recall})
    elapsed = time.time() - t0
    print(f"[{i+1}/{len(plant_groups)}] {held_out:20s} n_tiles={len(te_idx):2d}  "
          f"recall={recall:.3f}  (elapsed {elapsed:.0f}s)")

mean_recall = float(np.mean([r["recall"] for r in results]))
total_tiles = sum(r["n_tiles"] for r in results)
weighted_recall = sum(r["recall"] * r["n_tiles"] for r in results) / total_tiles

print(f"\n=== LOFO summary (N={len(results)} facilities) ===")
print(f"  unweighted mean recall: {mean_recall:.3f}")
print(f"  tile-weighted mean recall: {weighted_recall:.3f}")
worst = sorted(results, key=lambda r: r["recall"])[:5]
print(f"  worst-generalizing facilities: {[(r['plant'], round(r['recall'],2)) for r in worst]}")

lofo_out = {
    "epochs": EPOCHS,
    "n_facilities": len(results),
    "per_facility": results,
    "mean_recall": mean_recall,
    "tile_weighted_mean_recall": weighted_recall,
    "note": ("Each row trains a fresh model with that one facility's tiles fully "
             "excluded from training (all other plant facilities + all hard/rural "
             "negatives included). recall = fraction of the held-out facility's own "
             "tiles classified as 'plant'."),
}
lofo_path = f"{OUT_DIR}/lofo_track_a_results.json"
json.dump(lofo_out, open(lofo_path, "w"), indent=2)
print(f"\n[SAVED] {lofo_path}")

## 3b. LOFO + augmentation (same 20 folds, dihedral-4 spatial aug) -- from `lofo_track_a_aug.py`

Uses the baseline run's recall (just computed above) for the per-facility delta.

In [ ]:
baseline = {r["plant"]: r["recall"] for r in lofo_out["per_facility"]}

aug_results = []
t0 = time.time()
for i, held_out in enumerate(plant_groups):
    tr_idx = [j for j, g in enumerate(gp) if g != held_out]
    te_idx = [j for j, g in enumerate(gp) if g == held_out]
    Xtr = [Xp[j] for j in tr_idx] + Xh + Xr
    ytr = [yp[j] for j in tr_idx] + yh + yr
    Xte = [Xp[j] for j in te_idx]

    recall = train_and_eval(Xtr, ytr, Xte, seed=i, augment=True)
    base = baseline.get(held_out)
    aug_results.append({"plant": held_out, "n_tiles": len(te_idx), "recall": recall,
                         "baseline_recall": base,
                         "delta": (recall - base) if base is not None else None})
    elapsed = time.time() - t0
    delta_str = f"  delta={recall - base:+.2f}" if base is not None else ""
    print(f"[{i+1}/{len(plant_groups)}] {held_out:20s} n_tiles={len(te_idx):2d}  "
          f"recall={recall:.3f}{delta_str}  (elapsed {elapsed:.0f}s)")

aug_mean_recall = float(np.mean([r["recall"] for r in aug_results]))
aug_total_tiles = sum(r["n_tiles"] for r in aug_results)
aug_weighted_recall = sum(r["recall"] * r["n_tiles"] for r in aug_results) / aug_total_tiles

print(f"\n=== LOFO+aug summary (N={len(aug_results)} facilities) ===")
print(f"  unweighted mean recall: {aug_mean_recall:.3f}  (baseline: {mean_recall:.3f})")
print(f"  tile-weighted mean recall: {aug_weighted_recall:.3f}  (baseline: {weighted_recall:.3f})")

aug_out = {
    "epochs": EPOCHS,
    "n_facilities": len(aug_results),
    "per_facility": aug_results,
    "mean_recall": aug_mean_recall,
    "tile_weighted_mean_recall": aug_weighted_recall,
    "baseline_mean_recall": mean_recall,
    "baseline_tile_weighted_mean_recall": weighted_recall,
    "note": ("Same LOFO fold structure as the baseline run, but training tiles get a "
             "random dihedral-4 spatial transform (identity/hflip/vflip/90-rot) applied "
             "per-epoch via AugmentedTiles. Tests whether the baseline LOFO recall gap "
             "was partly a training-set-diversity artifact rather than purely a "
             "weak-signal problem."),
}
aug_path = f"{OUT_DIR}/lofo_track_a_aug_results.json"
json.dump(aug_out, open(aug_path, "w"), indent=2)
print(f"\n[SAVED] {aug_path}")

## 4. Download results back to your machine

Pulls both result JSONs down as a zip. Copy them into your local repo's `data/` to compare against (or replace, per the project's versioning convention -- don't overwrite silently) the M1-trained results.

In [ ]:
from google.colab import files
import shutil

os.makedirs('/content/results', exist_ok=True)
shutil.copy(lofo_path, '/content/results/')
shutil.copy(aug_path, '/content/results/')
shutil.make_archive('/content/lofo_track_a_results', 'zip', '/content/results')
files.download('/content/lofo_track_a_results.zip')